In [1]:
import aegnn
import torch

# Data visualization

In [2]:
import os
os.environ["AEGNN_DATA_DIR"] = "/home/kevin-imagine/Documents/event_graph/data"
print(os.environ["AEGNN_DATA_DIR"])

/home/kevin-imagine/Documents/event_graph/data


In [3]:
data_module = aegnn.datasets.NCars(batch_size=1, shuffle=False, )


In [4]:
data_module

NCars[Train: None
Validation: None]

In [5]:
data_module.setup()

In [6]:
data_module.train_dataset.files

['/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_7264',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_97',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_5024',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_1512',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_6704',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_1037',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_6195',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_9463',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_4835',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_11671',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_384',
 '/home/kevi

In [7]:
import glob
import os


In [8]:
data_module.root 

'/home/kevin-imagine/Documents/event_graph/data/ncars'

In [9]:
g = data_module.train_dataset[0]

In [10]:
g

Data(x=[10000, 1], pos=[10000, 3], file_id='sequence_7264', label=[1], y=[1], edge_index=[2, 315712], edge_attr=[315712, 3])

In [11]:
g.x.shape

torch.Size([10000, 1])

In [12]:
g.edge_index

tensor([[4906, 4584, 7533,  ...,  497, 8873, 4374],
        [   0,    0,    0,  ..., 9999, 9999, 9999]])

In [13]:
g.edge_attr

tensor([[0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        ...,
        [0.6667, 0.5000, 0.5000],
        [0.6667, 0.5000, 0.5000],
        [0.6667, 0.5000, 0.5000]])

In [14]:
g.label

['background']

In [15]:
g.y

tensor([1])

# Data pre-processing

 Convert event stream in event graph

## NCARS

In [16]:
from aegnn.datasets.ncars import NCars

In [17]:
ncars = NCars()

In [19]:
ncars

NCars[Train: None
Validation: None]

In [20]:
event_sample_file = ncars.raw_files("training")[0]
event_sample_file

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/training/*


'/home/kevin-imagine/Documents/event_graph/data/ncars/training/sequence_7264'

In [21]:
ncars.classes

['car', 'background']

In [22]:
ncars.pre_transform

<bound method NCars.pre_transform of NCars[Train: None
Validation: None]>

In [23]:
#Loading raw event data from file name
event_sample = ncars.load(event_sample_file)
event_sample

Data(x=[10066, 1], pos=[10066, 3])

On a ici environ 10000 events dans le raw event cloud.
Chaque event est un noeud, ayant une feature (x) avec la polarité et une position spatiale (pos) avec les coordonnées (x,y,t) 

In [24]:
event_sample.label = ncars.read_label(event_sample_file)
event_sample

Data(x=[10066, 1], pos=[10066, 3], label='background')

In [25]:
class_dict = {class_id: i for i, class_id in enumerate(ncars.classes)}
event_sample.y = torch.tensor([class_dict[event_sample.label]])
event_sample


Data(x=[10066, 1], pos=[10066, 3], label='background', y=[1])

In [26]:
event_sample_data_pt = ncars.pre_transform(event_sample)
event_sample_data_pt

Data(x=[10000, 1], pos=[10000, 3], label='background', y=[1], edge_index=[2, 315678])

### Pre-transform

In [27]:
from aegnn.datasets.utils.normalization import normalize_time
from torch_geometric.nn.pool import radius_graph

In [28]:
event_sample.pos

tensor([[6.0000e+00, 3.1000e+01, 0.0000e+00],
        [2.9000e+01, 1.9000e+01, 1.0000e-11],
        [3.3000e+01, 1.7000e+01, 2.0000e-11],
        ...,
        [1.6000e+01, 2.1000e+01, 4.9966e-07],
        [3.1000e+01, 5.2000e+01, 4.9966e-07],
        [4.3000e+01, 2.4000e+01, 4.9967e-07]])

In [29]:
event_sample.pos.shape

torch.Size([10066, 3])

In [30]:
torch.max(event_sample.pos[:,2])

tensor(4.9967e-07)

In [31]:
event_sample.pos[:, 2] = normalize_time(event_sample.pos[:, 2])
event_sample.pos

tensor([[6.0000e+00, 3.1000e+01, 0.0000e+00],
        [2.9000e+01, 1.9000e+01, 5.0000e-17],
        [3.3000e+01, 1.7000e+01, 1.0000e-16],
        ...,
        [1.6000e+01, 2.1000e+01, 2.4983e-12],
        [3.1000e+01, 5.2000e+01, 2.4983e-12],
        [4.3000e+01, 2.4000e+01, 2.4983e-12]])

La coordonnée temporelle a été normalisée 

In [32]:
params = ncars.hparams.preprocessing
event_subsampling = ncars.sub_sampling(event_sample, n_samples=params["n_samples"], sub_sample=params["sampling"])
event_subsampling

Data(x=[10000, 1], pos=[10000, 3], label='background', y=[1])

Le sub samplng de la class NCars permet de ramener le nombre d'events à exactement 10000. L'article indiquait un uniforme subsampling de fecteur K, ce qui n'est pas le cas ici. 

In [33]:
event_subsampling.edge_index = radius_graph(event_subsampling.pos, r=params["r"], max_num_neighbors=params["d_max"])
event_subsampling

Data(x=[10000, 1], pos=[10000, 3], label='background', y=[1], edge_index=[2, 315648])

Une fois le pre-transform réalisé, il est stocké dans un fichier. C'est donc ici que s'arrête le fichier de preprocessing.py

## NCaltech

In [16]:
from aegnn.datasets.ncaltech101 import NCaltech101

In [17]:
ncaltech = NCaltech101()

In [18]:
ncaltech

NCaltech101[Train: None
Validation: None]

In [25]:
event_sample_file = ncaltech.raw_files("training")[0]
event_sample_file

'/home/kevin-imagine/Documents/event_graph/data/ncaltech101/training/schooner/image_0051.bin'

In [26]:
ncaltech.classes

['schooner',
 'Faces_easy',
 'ketch',
 'inline_skate',
 'water_lilly',
 'garfield',
 'dalmatian',
 'stop_sign',
 'joshua_tree',
 'ceiling_fan',
 'dollar_bill',
 'ant',
 'pizza',
 'stegosaurus',
 'brontosaurus',
 'anchor',
 'airplanes',
 'helicopter',
 'cellphone',
 'crab',
 'watch',
 'lotus',
 'cougar_face',
 'revolver',
 'pigeon',
 'tick',
 'car_side',
 'binocular',
 'ferry',
 'crocodile',
 'Motorbikes',
 'elephant',
 'ewer',
 'snoopy',
 'gerenuk',
 'buddha',
 'platypus',
 'euphonium',
 'umbrella',
 'hawksbill',
 'lobster',
 'bonsai',
 'windsor_chair',
 'lamp',
 'cup',
 'laptop',
 'crayfish',
 'strawberry',
 'cannon',
 'rhino',
 'camera',
 'pagoda',
 'saxophone',
 'okapi',
 'scissors',
 'beaver',
 'trilobite',
 'wild_cat',
 'hedgehog',
 'sea_horse',
 'octopus',
 'headphone',
 'crocodile_head',
 'accordion',
 'flamingo_head',
 'dolphin',
 'mandolin',
 'kangaroo',
 'butterfly',
 'emu',
 'menorah',
 'brain',
 'stapler',
 'llama',
 'grand_piano',
 'electric_guitar',
 'cougar_body',
 'roos

### Load

In [25]:
import numpy as np

In [26]:
f = open(event_sample_file, 'rb')
raw_data = np.fromfile(f, dtype=np.uint8)
print(raw_data)
f.close()
raw_data = np.uint32(raw_data)

NameError: name 'event_sample_file' is not defined

In [51]:
all_y = raw_data[1::5]
all_x = raw_data[0::5]
all_p = (raw_data[2::5] & 128) >> 7 

In [52]:
raw_data[1::5]

array([139,  30,   2, ..., 136, 149, 130], dtype=uint32)

In [53]:
raw_data.shape

(401740,)

In [54]:
((raw_data[2::5] & 127) << 16).shape

(80348,)

In [55]:
((raw_data[3::5] << 8)).shape

(80348,)

In [56]:
(raw_data[4::5]).shape

(80348,)

In [57]:
all_ts = ((raw_data[2::5] & 127) << 16) | (raw_data[3::5] << 8) | (raw_data[4::5])

In [32]:
class_dict = {class_id: i for i, class_id in enumerate(ncaltech.classes)}

In [78]:
class_dict

{'schooner': 0,
 'Faces_easy': 1,
 'ketch': 2,
 'inline_skate': 3,
 'water_lilly': 4,
 'garfield': 5,
 'dalmatian': 6,
 'stop_sign': 7,
 'joshua_tree': 8,
 'ceiling_fan': 9,
 'dollar_bill': 10,
 'ant': 11,
 'pizza': 12,
 'stegosaurus': 13,
 'brontosaurus': 14,
 'anchor': 15,
 'airplanes': 16,
 'helicopter': 17,
 'cellphone': 18,
 'crab': 19,
 'watch': 20,
 'lotus': 21,
 'cougar_face': 22,
 'revolver': 23,
 'pigeon': 24,
 'tick': 25,
 'car_side': 26,
 'binocular': 27,
 'ferry': 28,
 'crocodile': 29,
 'Motorbikes': 30,
 'elephant': 31,
 'ewer': 32,
 'snoopy': 33,
 'gerenuk': 34,
 'buddha': 35,
 'platypus': 36,
 'euphonium': 37,
 'umbrella': 38,
 'hawksbill': 39,
 'lobster': 40,
 'bonsai': 41,
 'windsor_chair': 42,
 'lamp': 43,
 'cup': 44,
 'laptop': 45,
 'crayfish': 46,
 'strawberry': 47,
 'cannon': 48,
 'rhino': 49,
 'camera': 50,
 'pagoda': 51,
 'saxophone': 52,
 'okapi': 53,
 'scissors': 54,
 'beaver': 55,
 'trilobite': 56,
 'wild_cat': 57,
 'hedgehog': 58,
 'sea_horse': 59,
 'octopus

In [64]:
event_sample = ncaltech.load(event_sample_file)
event_sample

Data(x=[80348, 1], pos=[80348, 3])

In [65]:
label = ncaltech.read_label(event_sample_file)
label

'schooner'

In [66]:
event_sample.label = label if isinstance(label, list) else [label]
event_sample.y = torch.tensor([class_dict[label] for label in event_sample.label])

In [67]:
bbox = ncaltech.read_annotations(event_sample_file)
bbox

array([[[  5,   4, 173, 146,   0]]])

In [68]:
event_sample.bbox = torch.tensor(bbox, device="cuda").long()

In [69]:
event_sample

Data(x=[80348, 1], pos=[80348, 3], label=[1], y=[1], bbox=[1, 1, 5])

data_transform

In [70]:
window_us = 50 * 1000
t = event_sample.pos[event_sample.num_nodes // 2, 2]
t

tensor(0.1444, device='cuda:0')

In [71]:
data = event_sample
event_sample

Data(x=[80348, 1], pos=[80348, 3], label=[1], y=[1], bbox=[1, 1, 5])

In [73]:
num_nodes = data.num_nodes
num_nodes

80348

In [74]:
index1 = torch.clamp(torch.searchsorted(data.pos[:, 2].contiguous(), t) - 1, 0, data.num_nodes - 1)
index0 = torch.clamp(torch.searchsorted(data.pos[:, 2].contiguous(), t-window_us) - 1, 0, data.num_nodes - 1)

In [75]:
for key, item in data:
            if torch.is_tensor(item) and item.size(0) == num_nodes and item.size(0) != 1:
                data[key] = item[index0:index1, :]

In [76]:
torch.is_tensor(data.pos)

True

In [62]:
data.pos.size(0)

80348

In [77]:
data

Data(x=[40173, 1], pos=[40173, 3], label=[1], y=[1], bbox=[1, 1, 5])

In [79]:
params = ncaltech.hparams.preprocessing

In [80]:
data = ncaltech.sub_sampling(data, n_samples=params["n_samples"], sub_sample=params["sampling"])

In [81]:
data

Data(x=[25000, 1], pos=[25000, 3], label=[1], y=[1], bbox=[1, 1, 5])

Le dataset Ncaltech101, est à la fois un dataset de classification et de détection d'objets. Niveau annotation, il n'y a qu'une seule box, indépendente du temps, ce qui rend la tâche plus simple. Pour un sample, l'article choisie une fenêtre temporelle de 50 microsecondes. On récuper le temps de l'event T qui correspond à la moitié du sample. On prend comme index les indices de la fenêtre temporelle juste avant T. On réalise ensuite un subsampling sur 

In [20]:
processed_files = ncaltech.processed_files("training")
processed_files

['/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0051.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0041.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0005.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0057.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0036.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0026.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0058.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0016.bin',
 '/home/kevin-imagine/Documents/event_graph/data/ncaltech101/processed/training/schooner/image_0012.bin',
 '/home/kevin-imagine/Documents/event_graph/da

In [ ]:
ncaltech._load_dataset("training")

In [19]:
ncaltech.setup()

In [20]:
g = ncaltech.train_dataset[0]

In [22]:
g.bbox

tensor([[  5,   4, 173, 146,   0]])

# Training

## Dataset et Dataloader

In [34]:
from torch_geometric.transforms import Cartesian
from torch_geometric.data import Data
import torch_geometric

In [35]:
processed_files = ncars.processed_files("training")
processed_files

['/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_7264',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_97',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_5024',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_1512',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_6704',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_1037',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_6195',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_9463',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_4835',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_11671',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_384',
 '/home/kevi

In [36]:
event_processed_file = processed_files[0]
event_processed_file

'/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_7264'

In [37]:
event_processed_data = torch.load(event_processed_file, weights_only=False)
event_processed_data

Data(x=[10000, 1], pos=[10000, 3], file_id='sequence_7264', label=[1], y=[1], edge_index=[2, 315712])

Le dataset Ncars étant un dataset de classification, il n'y a pas de bounding box. Il n'y a pas de transform non plus.  

In [38]:
max_value = ncars.hparams.get("preprocessing", {}).get("r", None)
edge_attr_func = Cartesian(norm=True, max_value=max_value, cat=False)
edge_attr_func(event_processed_data)
event_processed_data

Data(x=[10000, 1], pos=[10000, 3], file_id='sequence_7264', label=[1], y=[1], edge_index=[2, 315712])

Le dataset est donc créé à partir de ces données.

## Create and run model


### Construction graphe

In [39]:
dataset = data_module.train_dataset

In [40]:
index = 0
num_events = 25000
radius = 3.0
max_num_neighbors = 32


In [41]:
sample = dataset[index % len(dataset)]
sample

Data(x=[10000, 1], pos=[10000, 3], file_id='sequence_7264', label=[1], y=[1], edge_index=[2, 315712], edge_attr=[315712, 3])

In [42]:
sample.pos = sample.pos[:, :2] #Pourquoi retirer la composante temporelle de la position?
sample

Data(x=[10000, 1], pos=[10000, 2], file_id='sequence_7264', label=[1], y=[1], edge_index=[2, 315712], edge_attr=[315712, 3])

In [43]:
data = Data(x=sample.x[:num_events], pos=sample.pos[:num_events])
data.batch = torch.zeros(data.num_nodes, device=data.x.device)
data.edge_index = torch_geometric.nn.radius_graph(data.pos, r=radius, max_num_neighbors=max_num_neighbors).long()
data.edge_attr = edge_attr_func(data).edge_attr
data

Data(x=[10000, 1], pos=[10000, 2], batch=[10000], edge_index=[2, 315712], edge_attr=[315712, 2])

In [44]:
index_new = min(num_events, sample.num_nodes - 1)
x_new = sample.x[index_new, :].view(1, -1)
pos_new = sample.pos[index_new, :2].view(1, -1)
event_new = Data(x=x_new, pos=pos_new, batch=torch.zeros(1, dtype=torch.long))
event_new

Data(x=[1, 1], pos=[1, 2], batch=[1])

`data` est le graphe initial. `event_new` se rajoute au graphe initial de manière asynchrone.

### Create model

In [45]:
input_shape = torch.tensor([*data_module.dims, data.pos.shape[-1]], device='cpu')
pooling_size = (10,10)

In [46]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)

In [47]:
model._modules.items()

odict_items([('conv1', SplineConv(1, 8, dim=2)), ('norm1', BatchNorm(8)), ('conv2', SplineConv(8, 16, dim=2)), ('norm2', BatchNorm(16)), ('conv3', SplineConv(16, 16, dim=2)), ('norm3', BatchNorm(16)), ('conv4', SplineConv(16, 16, dim=2)), ('norm4', BatchNorm(16)), ('conv5', SplineConv(16, 32, dim=2)), ('norm5', BatchNorm(32)), ('pool5', MaxPooling(voxel_size=[10, 10])), ('conv6', SplineConv(32, 32, dim=2)), ('norm6', BatchNorm(32)), ('conv7', SplineConv(32, 32, dim=2)), ('norm7', BatchNorm(32)), ('pool7', MaxPoolingX(voxel_size=tensor([30, 25]), size=16)), ('fc', Linear(in_features=512, out_features=2, bias=False))])

In [48]:
model

GraphRes(
  (conv1): SplineConv(1, 8, dim=2)
  (norm1): BatchNorm(8)
  (conv2): SplineConv(8, 16, dim=2)
  (norm2): BatchNorm(16)
  (conv3): SplineConv(16, 16, dim=2)
  (norm3): BatchNorm(16)
  (conv4): SplineConv(16, 16, dim=2)
  (norm4): BatchNorm(16)
  (conv5): SplineConv(16, 32, dim=2)
  (norm5): BatchNorm(32)
  (pool5): MaxPooling(voxel_size=[10, 10])
  (conv6): SplineConv(32, 32, dim=2)
  (norm6): BatchNorm(32)
  (conv7): SplineConv(32, 32, dim=2)
  (norm7): BatchNorm(32)
  (pool7): MaxPoolingX(voxel_size=tensor([30, 25]), size=16)
  (fc): Linear(in_features=512, out_features=2, bias=False)
)

In [49]:
model.to("cpu")

GraphRes(
  (conv1): SplineConv(1, 8, dim=2)
  (norm1): BatchNorm(8)
  (conv2): SplineConv(8, 16, dim=2)
  (norm2): BatchNorm(16)
  (conv3): SplineConv(16, 16, dim=2)
  (norm3): BatchNorm(16)
  (conv4): SplineConv(16, 16, dim=2)
  (norm4): BatchNorm(16)
  (conv5): SplineConv(16, 32, dim=2)
  (norm5): BatchNorm(32)
  (pool5): MaxPooling(voxel_size=[10, 10])
  (conv6): SplineConv(32, 32, dim=2)
  (norm6): BatchNorm(32)
  (conv7): SplineConv(32, 32, dim=2)
  (norm7): BatchNorm(32)
  (pool7): MaxPoolingX(voxel_size=tensor([30, 25]), size=16)
  (fc): Linear(in_features=512, out_features=2, bias=False)
)

### Make asynchronous

#### Modèle entier

In [50]:
log_flops=True
log_runtime=True

In [51]:
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)

In [52]:
async_model

GraphRes(
  (conv1): SplineConv(1, 8, dim=2)
  (norm1): BatchNorm(8)
  (conv2): SplineConv(8, 16, dim=2)
  (norm2): BatchNorm(16)
  (conv3): SplineConv(16, 16, dim=2)
  (norm3): BatchNorm(16)
  (conv4): SplineConv(16, 16, dim=2)
  (norm4): BatchNorm(16)
  (conv5): SplineConv(16, 32, dim=2)
  (norm5): BatchNorm(32)
  (pool5): MaxPooling(voxel_size=[10, 10])
  (conv6): SplineConv(32, 32, dim=2)
  (norm6): BatchNorm(32)
  (conv7): SplineConv(32, 32, dim=2)
  (norm7): BatchNorm(32)
  (pool7): MaxPoolingX(voxel_size=tensor([30, 25]), size=16)
  (fc): Linear(in_features=512, out_features=2, bias=False)
)

In [53]:
async_model.conv1.asy_pos

#### Asynchronous Conv

In [54]:
from torch_geometric.nn.conv import SplineConv

In [55]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)

In [56]:
spline_conv = SplineConv(1, 8, dim=2, kernel_size=2, bias=False, root_weight=False)

In [57]:
async_conv_layer = model.conv1
async_conv_layer

SplineConv(1, 8, dim=2)

In [58]:
# rajoute une mémoire du graphe à la couche de convolution
async_conv_layer.asy_graph = None
async_conv_layer.asy_flops_log = []
async_conv_layer.asy_runtime_log = []
async_conv_layer.asy_radius = radius

In [59]:
async_conv_layer.asy_pos = None

In [60]:
from aegnn.asyncronous.base.base import make_asynchronous

In [61]:
async_conv_layer.asy_is_initial = True #Pour la première couche de convoulution, sinon False
async_conv_layer.asy_edge_attributes = edge_attr_func #Fonction pour calculer les attributs
async_conv_layer.sync_forward = async_conv_layer.forward #Forward synchrone

Appel de la fonction make_synchronous() avec les arguments __graph_initialisation et __graph_processing

La fonction make_asynchronous permet de changer le forward du module en passant de synchrone à asynchrone. La fonction async_forward() appelle async_context() qui détermine quel func de graphe est appellé, suivant si on est sur le graphe d'initialisation ou sur un nouvel event. Cette fonction permet de calculer le temps d'execution pour le traitement d'un event. 

Une fois la fonction async_forward() définie comme module.forward(), une fonction de callback est défini. Celle-ci permet d'affecter à un attribut d'une couche suivante dans le modèle, la valeur de cette attribut pour la couche actuelle.

#### Forward

##### __graph_initialisation

In [62]:
#Lors de l'appel du forward, l'attribut pos du graphe passé en argument est assigné à l'attribut asy_pos de chaque couche (hors BatchNormalisation)
async_model.asy_pass_attribute('asy_pos', data.pos)

In [63]:
async_conv_layer = async_model.conv1
async_conv_layer.asy_edge_attributes

Cartesian(norm=True, max_value=3.0)

In [64]:
pos = async_conv_layer.asy_pos
pos

tensor([[24., 58.],
        [46., 31.],
        [ 6., 35.],
        ...,
        [14.,  0.],
        [ 0., 43.],
        [27., 16.]])

On suppose ici que les edge_attributs et les egdes index sont donnés à la couche (ce qui est le cas dans le code) 

In [65]:
#Le graphe initiale est inféré avec la fonction forward synchrone
y = async_conv_layer.sync_forward(data.x, edge_index=data.edge_index, edge_attr=data.edge_attr)
y.shape

/home/kevin-imagine/Documents/event_graph/.venv/lib/python3.10/site-packages/torch_geometric/nn/conv/spline_conv.py:133: UserWarning: We do not recommend using the non-optimized CPU version of `SplineConv`. If possible, please move your data to GPU.
  warnings.warn(


torch.Size([10000, 8])

In [66]:
async_conv_layer.asy_graph = Data(x=data.x, pos=pos, edge_index=data.edge_index, edge_attr=data.edge_attr,y=y)
async_conv_layer.asy_graph

Data(x=[10000, 1], edge_index=[2, 315712], edge_attr=[315712, 2], y=[10000, 8], pos=[10000, 2])

Une instruction permet de calculer le nombre de flops nécessaire à l'inférence du graphe d'initialisation

In [67]:
#Fonction qui permet d'assigner pos à l'attribut asy_pos des couches suivantes dans le réseau
#Semble faire doublon avec 
async_conv_layer.asy_pass_attribute('asy_pos', pos)

#### __graph_processing

Une fois le graph d'initialisation inféré, c'est la fonction __graph_processing qui est appelé pour le forward. La fonction prend en argument le module (la couche elle même), x (les features de la nouvelle node), edge_index et edge_attributes 

##### Conv1 couche initial

In [68]:
async_model.asy_pass_attribute('asy_pos', event_new.pos)

In [69]:
async_conv_layer.asy_pos

tensor([[27., 16.]])

In [70]:
pos = async_conv_layer.asy_pos

In [71]:
print(f"Input graph with x = {event_new.x.shape} and pos = {pos.shape}")
print(f"Internal graph = {async_conv_layer.asy_graph}")

Input graph with x = torch.Size([1, 1]) and pos = torch.Size([1, 2])
Internal graph = Data(x=[10000, 1], edge_index=[2, 315712], edge_attr=[315712, 2], y=[10000, 8], pos=[10000, 2])


On étudie la couche initiale ici (conv1)

In [72]:
num_prev_nodes = async_conv_layer.asy_graph.num_nodes
num_prev_nodes

10000

In [73]:
x_new, pos_new = event_new.x, pos

In [74]:
#Détermine les index des nouvelles nodes après concaténation  
idx_new = torch.arange(num_prev_nodes, num_prev_nodes + event_new.x.size()[0], device=event_new.x.device)
print(idx_new)

tensor([10000])


In [75]:

idx_diff = torch.tensor([], device=event_new.x.device, dtype=torch.long)
idx_diff

tensor([], dtype=torch.int64)

In [76]:
x_all = torch.cat([async_conv_layer.asy_graph.x, x_new], dim=0)
pos_all = torch.cat([async_conv_layer.asy_graph.pos, pos_new], dim=0)

In [77]:
print(f"Subgraph contains {idx_new.numel()} new and {idx_diff.numel()} diff nodes")

Subgraph contains 1 new and 0 diff nodes


In [78]:
#Détermine les nodes existants qui sont connectés avec la nouvelle node
connected_node_mask = torch.cdist(pos_all, pos_new) <= async_conv_layer.asy_radius
connected_node_mask.sum()

tensor(314)

In [79]:
torch.cdist(pos_all, pos_new) 

tensor([[42.1070],
        [24.2074],
        [28.3196],
        ...,
        [38.1838],
        [ 0.0000],
        [ 0.0000]])

In [80]:
idx_new_neigh = torch.unique(torch.nonzero(connected_node_mask)[:, 0])
idx_new_neigh #Index des voisins de la nouvelle node

tensor([   15,   108,   140,   142,   143,   145,   190,   247,   278,   301,
          334,   362,   402,   443,   475,   487,   491,   497,   513,   524,
          550,   605,   619,   629,   710,   739,   765,   801,   877,   892,
          902,   906,   929,   980,   993,   995,  1017,  1020,  1072,  1104,
         1142,  1151,  1155,  1181,  1205,  1206,  1227,  1241,  1244,  1280,
         1299,  1332,  1339,  1350,  1370,  1398,  1441,  1468,  1474,  1490,
         1510,  1520,  1527,  1532,  1559,  1652,  1666,  1703,  1712,  1745,
         1757,  1773,  1778,  1794,  1853,  1889,  1941,  1980,  2036,  2164,
         2190,  2203,  2271,  2280,  2400,  2402,  2410,  2444,  2452,  2462,
         2507,  2519,  2525,  2541,  2601,  2685,  2856,  2922,  2942,  2953,
         2956,  2970,  2987,  2988,  3018,  3063,  3185,  3222,  3321,  3404,
         3442,  3451,  3469,  3484,  3489,  3504,  3519,  3561,  3655,  3678,
         3704,  3731,  3749,  3793,  3815,  3873,  3882,  3904, 

In [81]:
idx_update = torch.cat([idx_new_neigh, idx_diff]) 
#Les index des noeuds impactés depuis le dernier forward (noeuds dont les features ont changé (aucune a priori) et ceux recevant un nouveau voisin)

In [82]:
idx_update.shape

torch.Size([314])

In [83]:
from torch_geometric.utils import k_hop_subgraph
# récupère les voisins des voisins de la nouvelle node
_, edges_connected, _, connected_edges_mask = k_hop_subgraph(idx_update, num_hops=1,
                                                                 edge_index=async_conv_layer.asy_graph.edge_index,
                                                                 num_nodes=pos_all.shape[0])

In [84]:
edges_new = torch.nonzero(connected_node_mask).T
edges_new[1, :] = idx_new[edges_new[1,:]] #changement d'index pour ramener à l'index du graphe global

In [85]:
edges_new_inv = torch.stack([edges_new[1, :], edges_new[0, :]], dim=0)
edges_new_inv.shape

torch.Size([2, 314])

In [86]:
edges_new = torch.cat([edges_new, edges_new_inv], dim=1)
edges_new.shape
#Permet d'obtenir un graphe non orienté

torch.Size([2, 628])

In [87]:
edges_new = torch.unique(edges_new, dim=1)
edges_new.shape

torch.Size([2, 627])

In [88]:
from torch_geometric.utils import remove_self_loops
edges_new, _ = remove_self_loops(edges_new)

In [89]:
edge_index = torch.cat([edges_connected, edges_new], dim=1) # On obtient les edges index d'un subgraph de k = 2 à partir de la nouvelle node

In [90]:
async_conv_layer.asy_edge_attributes

Cartesian(norm=True, max_value=3.0)

In [91]:
#Calul des edges_attributes
graph_new = Data(x=x_all, pos=pos_all, edge_index=edges_new)
edge_attr_new = async_conv_layer.asy_edge_attributes(graph_new).edge_attr
edge_attr_connected = async_conv_layer.asy_graph.edge_attr[connected_edges_mask, :]
edge_attr = torch.cat([edge_attr_connected, edge_attr_new])
edge_attr

tensor([[0.5000, 0.5000],
        [0.5000, 0.5000],
        [0.5000, 0.5000],
        ...,
        [0.3333, 0.8333],
        [0.6667, 0.8333],
        [0.5000, 0.5000]])

In [92]:
x_new

tensor([[0.]])

In [93]:
out_channels = async_conv_layer.asy_graph.y.size()[-1]
y = torch.cat([async_conv_layer.asy_graph.y.clone(), torch.zeros(x_new.size()[0], out_channels, device=event_new.x.device)])

In [94]:
x_j = x_all[edge_index[0, :], :] #Features des voisins et de distance k-hops = 2
x_j

tensor([[0.],
        [0.],
        [0.],
        ...,
        [0.],
        [0.],
        [0.]])

In [95]:
#Calcule du message 
phi = async_conv_layer.message(x_j, edge_attr=edge_attr)

In [96]:
y_update = async_conv_layer.aggregate(phi, index=edge_index[1,:], ptr=None, dim_size=x_all.size()[0])
y_update

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0003, 0.0042, 0.0040,  ..., 0.0006, 0.0003, 0.0037]],
       grad_fn=<DivBackward0>)

In [97]:
y[idx_update] = y_update[idx_update]

In [98]:
print(f"Updated {idx_update.numel()} nodes in asy. graph of module {async_conv_layer}")

Updated 314 nodes in asy. graph of module SplineConv(1, 8, dim=2)


Il y a possibilté de calculer le nomnbre de flops si besoin. Si la couche est une couche initiale (comme conv1), on assigne à l'attribut asy_pos des autres couches la valeur pos_all 

##### Conv2

In [99]:
data

Data(x=[10000, 1], pos=[10000, 2], batch=[10000], edge_index=[2, 315712], edge_attr=[315712, 2])

In [100]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)

In [101]:
_ = async_model(data)

In [102]:
event_new

Data(x=[1, 1], pos=[1, 2], batch=[1])

In [103]:
from torch.nn.functional import elu

In [104]:
async_model.asy_pass_attribute('asy_pos', event_new.pos)
x_f = elu(async_model.conv1(event_new.x))
x_f = async_model.norm1(x_f)

In [105]:
async_conv2_layer = async_model.conv2

On se retouve maintenant dans la partie qui concerne les couches de convolution qui ne sont pas la première

In [106]:
pos = async_conv2_layer.asy_pos

In [107]:
print(f"Input graph with x = {x_f.shape} and pos = {pos.shape}")
print(f"Internal graph = {async_conv2_layer.asy_graph}")

Input graph with x = torch.Size([10001, 8]) and pos = torch.Size([10001, 2])
Internal graph = Data(x=[10000, 8], edge_index=[2, 315712], edge_attr=[315712, 2], y=[10000, 16], pos=[10000, 2])


Pour la couche de convolution 2, l'input est la sortie de la conv1, avec donc donc 10001 nodes de 8 channels chacun.

In [108]:
from aegnn.asyncronous.base.utils import graph_new_nodes, graph_changed_nodes

In [109]:
#On récupère la nouvelle node et son index dans le graphe
x_new, idx_new = graph_new_nodes(async_conv2_layer, x=x_f)

In [110]:
pos_new = pos[idx_new, :] #La position de la nouvelle node

In [111]:
_, idx_diff = graph_changed_nodes(async_conv2_layer, x=x_f) #On récupère les index des nodes qui ont changé (qui ont été impacté par l'ajout de la nouvelle node dans la conv1)

In [112]:
idx_diff

tensor([ 142,  145,  475,  487,  497,  550,  980,  993, 1072, 1104, 1151, 1559,
        1652, 1666, 1778, 1794, 1889, 1980, 2164, 2190, 2400, 2410, 2856, 2922,
        2956, 2970, 3018, 3063, 3222, 3442, 3484, 3655, 3678, 3704, 3731, 3793,
        3882, 4071, 4077, 4179, 4321, 4374, 5505, 5515, 5862, 6036, 6254, 6289,
        6685, 6790, 6802, 7145, 7326, 7487, 7572, 7604, 7971, 8200, 8465, 8526,
        8873, 9032, 9048, 9087, 9323, 9326, 9552, 9600, 9763])

In [113]:
#On récupère les indexs des voisins des noeuds qui ont changés
idx_diff, _, _, _ = k_hop_subgraph(idx_diff, num_hops=1, edge_index=async_conv2_layer.asy_graph.edge_index,
                                               num_nodes=async_conv2_layer.asy_graph.num_nodes + len(idx_new))

La suite est le même traitement pour trouver les edge_index et les edge_attr

### Etude de la couche Batch Normalization

Les couches de BN peuvent être rendu asynchrone. Contrairement aux autres types de couches, celles-ci n'ont pas de callback et ne sont pas concernés par le callback des autres.

In [114]:
from torch.nn.functional import elu

In [115]:
data

Data(x=[10000, 1], pos=[10000, 2], batch=[10000], edge_index=[2, 315712], edge_attr=[315712, 2])

In [116]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)

In [117]:
async_bn_layer = model.norm1

In [118]:
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)

In [119]:
async_model.asy_pass_attribute('asy_pos', data.pos)

In [120]:
x_f = elu(async_model.conv1(data.x, data.edge_index, data.edge_attr))

In [121]:
async_bn_layer.asy_graph = None
async_bn_layer.asy_flops_log = [] if log_flops else None
async_bn_layer.asy_runtime_log = [] if log_runtime else None


#### __graph_initiatization

In [122]:
#On calcule la moyenne et la variance du graphe initiale
mean = torch.mean(x_f, dim = 0)
var = torch.var(x_f, dim = 0) + async_bn_layer.module.eps

In [123]:
async_bn_layer.asy_graph = torch_geometric.data.Data(x=mean,variance=var)
#La moyenne et la variance sont stocké dans un graphe

#### _graph_processing

In [124]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)
_ = async_model(data)

In [125]:
async_bn_layer = async_model.norm1

In [126]:
async_model.asy_pass_attribute('asy_pos', event_new.pos)
x_f = elu(async_model.conv1(event_new.x))

In [127]:
x_f.shape

torch.Size([10001, 8])

In [128]:
y = (x_f - async_bn_layer.asy_graph.x) / async_bn_layer.asy_graph.variance * async_bn_layer.module.weight + async_bn_layer.module.bias

On normalise puis on scale les features des noeuds du graphe en entrée

### Etude de la couche linéaire

In [129]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)
_ = async_model(data)

In [130]:
async_model.feature_map.shape

torch.Size([1, 512])

In [131]:
async_model(event_new)

tensor([[-32.7765,  -6.9884]], grad_fn=<CopySlices>)

In [132]:
async_linear_layer = async_model.fc
feature_map = async_model.feature_map

In [133]:
async_linear_layer.asy_graph

Data(x=[1, 512], y=[1, 2])

In [134]:
feature_map.shape

torch.Size([1, 512])

In [135]:
diff_idx = torch.nonzero(feature_map - async_linear_layer.asy_graph.x, as_tuple=True)

In [136]:
len(diff_idx) * diff_idx[0].shape[0]

0

In [137]:
diff_idx2 = torch.nonzero(feature_map - async_linear_layer.asy_graph.x).t().detach().cpu().numpy()
diff_idx2.shape[0] * diff_idx2.shape[1] 

0

In [138]:
feature_map[diff_idx]

tensor([], grad_fn=<IndexBackward0>)

In [139]:
async_linear_layer.asy_graph.x[diff_idx]

tensor([], grad_fn=<IndexBackward0>)

In [140]:
x_diff = feature_map[diff_idx] - async_linear_layer.asy_graph.x[diff_idx]

In [141]:
#On calcule un résidu sur les features qui ont changés depuis l'ajout d'un nouvel event
y_residual = torch.mul(async_linear_layer.weight[:, diff_idx[1]], x_diff).t()

In [142]:
async_linear_layer.weight.shape

torch.Size([2, 512])

In [143]:
#update
async_linear_layer.asy_graph.x[diff_idx] = feature_map[diff_idx]


In [144]:
async_linear_layer.asy_graph.y[diff_idx[0],:] += y_residual

In [145]:
async_linear_layer.asy_graph.y

tensor([[-32.7765,  -6.9884]], grad_fn=<CopySlices>)

In [146]:
flops = int(diff_idx[0].shape[0] * diff_idx[1].shape[0])
flops

0

In [147]:
diff_idx[0].shape[0]

0

In [148]:
diff_idx[1].shape

torch.Size([0])

### Etude des couches de pooling et de l'asynchronicité

Dans le modèle AEGNN, il y a 2 types de couches pour le max pooling.
- max_pooling: des voxels spaciaux sont créés à partir de la position des noeuds et du voxel size puis un max pooling est réalisé sur chaque cluster. A noter que la composante temporelle ne fait pas partie de la clusterisation. 
- max_pooling_x : 

##### max_pooling layer

In [149]:
from torch_geometric.nn.pool import max_pool, voxel_grid

In [150]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)
_ = async_model(data)

In [151]:
data_before_pool= async_model.fm_to_pool

In [152]:
data_before_pool

Data(x=[10000, 32], edge_index=[2, 315712], edge_attr=[315712, 2], pos=[10000, 2], batch=[10000])

In [153]:
cluster = voxel_grid(data_before_pool.pos[:, :2], batch = data_before_pool.batch, size = pooling_size)
cluster.shape

torch.Size([10000])

In [154]:
torch.unique(cluster, return_counts=True)

(tensor([ 0,  1,  2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
         19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]),
 tensor([  1, 301,   4, 103, 638, 599, 710, 785, 334, 861, 295, 326, 322, 336,
         810, 275, 266, 287, 240, 434,  44,  40,  52,  92, 373, 643, 336, 266,
         227]))

In [155]:
k = 1 #ID du voxel
mask = (cluster == k)

nodes_voxel_index = mask.nonzero(as_tuple=True)[0]
nodes_voxel_index.shape

torch.Size([301])

In [156]:
voxel_graph = Data(x=data_before_pool.x[nodes_voxel_index], pos=data_before_pool.pos[nodes_voxel_index])
voxel_graph

Data(x=[301, 32], pos=[301, 2])

In [157]:
max_pooled_features = torch.max(voxel_graph.x, dim=0)
max_pooled_features

torch.return_types.max(
values=tensor([16.2208, 26.2831, 21.3242, 18.8833, 11.5454, 17.7644, 28.4752, 10.0765,
        16.0865, 19.4614, 24.0271, 27.1238, 24.7543, 19.4208,  7.5912,  2.0817,
        14.7113,  9.2650, 18.4596, 29.5529, -1.7825, 14.0600, 24.1226,  9.8835,
        60.6269, 25.1525, 14.8999,  5.2461, 15.5529,  8.0962, 13.8667, 35.0324],
       grad_fn=<MaxBackward0>),
indices=tensor([260, 189,  41, 229,  40,  44,  18, 282, 119, 282,  41,  28,  30, 154,
         91,  28, 212,   4, 282,  92, 262,  30, 295, 116,  30, 143,  64,  40,
        128,  19,  58,  41]))

In [158]:
transform = Cartesian(norm=True, cat=False)

In [159]:
data_after_pool = max_pool(cluster, data=data_before_pool, transform=transform)
data_after_pool

DataBatch(x=[29, 32], edge_index=[2, 121], edge_attr=[121, 2], pos=[29, 2], batch=[29])

In [160]:
unique_clusters = torch.unique(cluster)

idx = (unique_clusters == k).nonzero(as_tuple=True)[0].item()
idx

1

In [161]:
pooled_nodes_features = data_after_pool.x[idx]

In [162]:
torch.max(max_pooled_features.values - pooled_nodes_features)

tensor(0., grad_fn=<MaxBackward1>)

Pour chaque cluster, les features de la pooled node sont calculés en gardant le max sur chaque canal indépendement en prenant en compte les features des nodes contituant le cluster. 

In [163]:
max_pool_pos = torch.mean(voxel_graph.pos, dim=0)
max_pool_pos

tensor([12.6047,  4.4452])

In [164]:
torch.max(max_pool_pos - data_after_pool.pos[idx])

tensor(0.)

Pour chaque cluster, la position de la pooled node est calculée en faisant la moyenne des positions des nodes qui composent le cluster

Pour les edges_index, on considère un edge entre 2 clusters si une node de chaque cluster entre elles avaient un edge.
Les edges_attributes sont calculés ensuites comme pour les autres graphes

##### max_pooling_x

In [165]:
from torch_geometric.nn.pool import max_pool_x

In [166]:
data_before_pool_x = async_model.fm_to_pool_x

In [167]:
voxel_size = input_shape[:2] // 4
voxel_size

tensor([30, 25])

In [168]:
cluster = voxel_grid(data_before_pool_x.pos[:,:2], batch=data_before_pool_x.batch, size=voxel_size)
cluster

tensor([0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 2, 2, 2, 3, 3, 2, 2, 2, 3, 3,
        4, 4, 4, 5, 5])

In [169]:
torch.unique(cluster, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5]), tensor([9, 5, 6, 4, 3, 2]))

In [170]:
#Seuls les features des noeuds sont renvoyés
#L'argument size permet d'obtenir le nombre de clusters renvoyés. Ici, Le nombre de cluster demandé (16) est plus grand que le nombre de cluster renvoyés par voxel grid
x_pooled, _ = max_pool_x(cluster, data_before_pool_x.x, data_before_pool_x.batch, size=16)
x_pooled.shape

torch.Size([16, 32])

Cette seconde couche de pooling est une version simplifié de la première qui ne renvoie que les features.

### Max pool asynchrone

In [171]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)

In [172]:
async_mp_layer = model.pool5

In [173]:
async_mp_layer.asy_graph = None
async_mp_layer.asy_flops_log = [] if log_flops else None
async_mp_layer.asy_runtime_log = [] if log_runtime else None
async_mp_layer.asy_radius = radius

In [174]:
async_mp_layer.asy_pos = None
async_mp_layer.asy_graph_coarse = None  # coarse output graph
async_mp_layer.asy_node_max_index = None  # index of max. node in input data
async_mp_layer.asy_voxel_pos_sum = None  # sum of positions per voxel
async_mp_layer.asy_voxel_node_count = None  # count of nodes per voxel

In [175]:
async_mp_layer.grid_size = list(data_module.dims)# grid size in N dimensions

In [176]:
async_mp_layer.grid_size

[120, 100]

La couche de max pooling garde en mémoire le coarse graph.

#### __graph_initialization

In [177]:
from aegnn.asyncronous.max_pool import __dense_process, __get_clusters
from torch_scatter import scatter_max, scatter_sum

In [178]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)

In [179]:
async_mp_layer = async_model.pool5

In [180]:
async_model.asy_pass_attribute('asy_pos', data.pos)

In [181]:
x_f = data.x.clone()
x_f = elu(async_model.conv1(x_f, data.edge_index, data.edge_attr))
x_f = async_model.norm1(x_f)
x_f = elu(async_model.conv2(x_f, data.edge_index, data.edge_attr))
x_f = async_model.norm2(x_f)

x_sc = x_f.clone()
x_f = elu(async_model.conv3(x_f, data.edge_index, data.edge_attr))
x_f = async_model.norm3(x_f)
x_f = elu(async_model.conv4(x_f, data.edge_index, data.edge_attr))
x_f = async_model.norm4(x_f)
x_f = x_f + x_sc

x_f = elu(async_model.conv5(x_f, data.edge_index, data.edge_attr))
x_f = async_model.norm5(x_f)

In [182]:
fm_to_pool = Data(x=x_f, pos=data.pos, batch=data.batch, edge_index=data.edge_index, edge_attr=data.edge_attr)

In [183]:
print(f"Input graph with x = {fm_to_pool.x.shape} and pos = {fm_to_pool.pos.shape}")

Input graph with x = torch.Size([10000, 32]) and pos = torch.Size([10000, 2])


In [184]:
graph_out = __dense_process(async_mp_layer, fm_to_pool.x, fm_to_pool.pos, edge_index=None)
graph_out #On obtient le coarse graphe comme dans le forward de la partie synchro

DataBatch(x=[29, 32], edge_index=[2, 142], pos=[29, 2], batch=[29], edge_attr=[142, 2], vdx=[29])

In [185]:
async_mp_layer.asy_graph = Data(x=fm_to_pool.x, pos=fm_to_pool.pos) #On stock le graphe entier

In [186]:
async_mp_layer.asy_graph.vdx = __get_clusters(async_mp_layer, pos=fm_to_pool.pos)  # voxel index of every node

In [187]:
async_mp_layer.asy_graph.vdx

tensor([67, 43, 39,  ...,  1, 52, 15])

In [188]:
a , index = torch.unique(async_mp_layer.asy_graph.vdx, sorted=True, return_inverse=True)

In [189]:
index.shape

torch.Size([10000])

In [190]:
_, argmax = scatter_max(fm_to_pool.x, index=index, dim=0)

In [191]:
argmax.shape #Pour chaque chaque feature de chaque voxel node, on garde l'index de la node du graphe qui y a donné la valeur

torch.Size([29, 32])

In [192]:
argmax_nodes = torch.unique(argmax)
async_mp_layer.asy_node_max_index = torch.flatten(argmax_nodes).long()
async_mp_layer.asy_node_max_index
#on stock les index des noeuds qui ont contribué à obtenir les features des voxel_nodes

tensor([   1,    3,    4,    7,    8,   10,   16,   21,   47,   51,   54,   55,
          59,   76,   86,   89,   96,  106,  111,  116,  125,  140,  141,  142,
         145,  147,  151,  159,  168,  173,  178,  183,  197,  200,  205,  209,
         211,  213,  220,  239,  240,  250,  253,  263,  277,  279,  295,  312,
         320,  322,  326,  331,  332,  342,  344,  350,  359,  380,  381,  385,
         392,  393,  399,  424,  433,  436,  448,  450,  451,  454,  457,  472,
         478,  481,  491,  501,  529,  541,  547,  551,  570,  593,  603,  617,
         627,  632,  667,  682,  688,  690,  696,  711,  714,  729,  764,  774,
         777,  783,  789,  792,  795,  804,  808,  815,  821,  825,  833,  838,
         841,  852,  863,  865,  867,  912,  920,  938,  949,  951,  961,  971,
         973,  985,  999, 1004, 1011, 1018, 1024, 1025, 1040, 1047, 1049, 1053,
        1062, 1070, 1078, 1089, 1090, 1111, 1128, 1140, 1145, 1155, 1160, 1175,
        1187, 1203, 1218, 1229, 1232, 12

In [193]:
# On calcule la somme des positions pour chaque voxel
async_mp_layer.asy_voxel_pos_sum = scatter_sum(fm_to_pool.pos, index=index, dim=0)
#On stocke le nombre de nodes par voxel
async_mp_layer.asy_voxel_node_count = scatter_sum(torch.ones_like(async_mp_layer.asy_graph.vdx, device=fm_to_pool.x.device), index=index)


In [194]:
async_mp_layer.asy_voxel_node_count

tensor([  1, 301,   4, 103, 638, 599, 710, 785, 334, 861, 295, 326, 322, 336,
        810, 275, 266, 287, 240, 434,  44,  40,  52,  92, 373, 643, 336, 266,
        227])

In [195]:
async_mp_layer.asy_graph_coarse = graph_out.clone()
#Le graphe after pooling est aussi stocké. 

##### __graph processing

In [196]:
model = aegnn.models.networks.GraphRes(data_module.name, input_shape, data_module.num_classes, pooling_size=pooling_size)
async_model = aegnn.asyncronous.make_model_asynchronous(model, radius, list(data_module.dims), edge_attr_func, log_flops=log_flops, log_runtime=log_runtime)
_ = async_model(data)

In [197]:
async_model.asy_pass_attribute('asy_pos', event_new.pos)

In [198]:
event_new.pos

tensor([[27., 16.]])

In [199]:
async_model.asy_pass_attribute.listeners

[SplineConv(1, 8, dim=2),
 SplineConv(8, 16, dim=2),
 SplineConv(16, 16, dim=2),
 SplineConv(16, 16, dim=2),
 SplineConv(16, 32, dim=2),
 MaxPooling(voxel_size=[10, 10]),
 SplineConv(32, 32, dim=2),
 SplineConv(32, 32, dim=2),
 Linear(in_features=512, out_features=2, bias=False)]

In [200]:
x_f = event_new.x.clone()
x_f = elu(async_model.conv1(x_f, event_new.edge_index, event_new.edge_attr))
x_f = async_model.norm1(x_f)
x_f = elu(async_model.conv2(x_f, event_new.edge_index, event_new.edge_attr))
x_f = async_model.norm2(x_f)

x_sc = x_f.clone()
x_f = elu(async_model.conv3(x_f, event_new.edge_index, event_new.edge_attr))
x_f = async_model.norm3(x_f)
x_f = elu(async_model.conv4(x_f, event_new.edge_index, event_new.edge_attr))
x_f = async_model.norm4(x_f)
x_f = x_f + x_sc

x_f = elu(async_model.conv5(x_f, event_new.edge_index, event_new.edge_attr))
x_f = async_model.norm5(x_f)

In [201]:
async_mp_layer = async_model.pool5

In [202]:
fm_to_pool = Data(x=x_f, pos=event_new.pos, batch=event_new.batch, edge_index=event_new.edge_index, edge_attr=event_new.edge_attr)

In [203]:
pos = async_mp_layer.asy_pos
pos.shape

torch.Size([10001, 2])

In [204]:
_, diff_idx = graph_changed_nodes(async_mp_layer, x= fm_to_pool.x)
_, new_idx = graph_new_nodes(async_mp_layer, x=fm_to_pool.x)

In [205]:
print(f"Subgraph contains {new_idx.numel()} new and {diff_idx.numel()} diff nodes")

Subgraph contains 1 new and 409 diff nodes


In [206]:
from aegnn.asyncronous.max_pool import __intersection

In [207]:
#On détermine si les nodes qui ont changés ont contribué au coarse graph
replaced_idx = __intersection(diff_idx.long(), async_mp_layer.asy_node_max_index)
print(f"... from which {replaced_idx.numel()} nodes contributed to the coarse graph")

... from which 9 nodes contributed to the coarse graph


In [208]:
torch.unique(replaced_idx, return_counts=True)

(tensor([ 108,  140,  475,  550,  801, 1155, 1227, 1726, 7969]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1]))

Si une node, qui a contribué à un voxel node, a changé, alors il faut reprendre toutes les nodes du voxel pour calculer à nouveau la voxel node.

In [209]:
graph_vdx = getattr(async_mp_layer.asy_graph, "vdx")

In [210]:
for idx in replaced_idx:
    nodes_voxel = torch.nonzero(torch.eq(graph_vdx, graph_vdx[idx]))[:, 0]
    #On détermine les nodes du voxel auquel appartient la node contributrice qui a changé de features
    diff_idx = torch.cat([diff_idx, nodes_voxel])

    voxel_idx = async_mp_layer.asy_graph.vdx[idx]
    coarse_idx = torch.eq(getattr(async_mp_layer.asy_graph_coarse, "vdx"), voxel_idx)

    async_mp_layer.asy_graph_coarse.x[coarse_idx] = -999999 
    async_mp_layer.asy_voxel_pos_sum[coarse_idx] -= pos[idx]
    async_mp_layer.asy_voxel_node_count[coarse_idx] -= 1


In [211]:
diff_idx.shape

torch.Size([6415])

In [ ]:
#On récupère les features et positions des nodes qui ont changé 
update_idx = torch.cat([diff_idx, new_idx])
x_update = x_f[update_idx, :]
pos_update = pos[update_idx, :]

In [213]:
update_idx.shape

torch.Size([6416])

In [214]:
vdx_update = __get_clusters(async_mp_layer, pos=pos_update)

In [218]:
async_mp_layer.asy_graph_coarse.x.shape
x_update.shape

torch.Size([6416, 32])

In [ ]:
#On concatène les update avec les variables en mémoire 
x_scatter = torch.cat([x_update, async_mp_layer.asy_graph_coarse.x])
node_index_scatter = torch.cat([update_idx, async_mp_layer.asy_node_max_index])
pos_sum_scatter = torch.cat([pos_update, async_mp_layer.asy_voxel_pos_sum])
node_count_scatter = torch.cat([torch.ones_like(update_idx, device=x_f.device), async_mp_layer.asy_voxel_node_count])
clusters = torch.cat([vdx_update, getattr(async_mp_layer.asy_graph_coarse, "vdx")])

In [ ]:
#On calcule le max sur chaque cluster 
clusters_unique, index = torch.unique(clusters, sorted=True, return_inverse=True)
x_max, argmax = scatter_max(x_scatter, index=index, dim=0)

In [225]:
torch.unique(argmax, return_counts=True)

(tensor([   1,    2,   18,   22,   25,   53,   85,  312,  421,  436,  460,  463,
          471,  472,  519,  521,  546,  561,  872,  941, 6092, 6101, 6111, 6126,
         6130, 6141, 6142, 6152, 6172, 6177, 6178, 6179, 6185, 6193, 6210, 6269,
         6320, 6358, 6416, 6417, 6418, 6419, 6420, 6421, 6423, 6424, 6425, 6426,
         6428, 6429, 6430, 6431, 6432, 6433, 6434, 6435, 6436, 6437, 6438, 6439,
         6440, 6441, 6442, 6443, 6444]),
 tensor([ 1,  1,  1,  1,  1,  1,  1,  2,  1,  1,  1,  1,  4,  2,  2,  1,  4,  1,
          5,  2,  2,  2,  1,  1,  2,  1,  2,  2,  4,  1,  1,  1,  1,  1,  1,  2,
          1,  4, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32,
         32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32]))

In [ ]:
#On calcule la position des voxels nodes
voxel_pos_sum = scatter_sum(pos_sum_scatter, index=index, dim=0)
voxel_node_count = scatter_sum(node_count_scatter, index=index)
pos_mean = torch.div(voxel_pos_sum, voxel_node_count.view(-1, 1))

In [227]:
vdx = torch.cat([getattr(async_mp_layer.asy_graph, "vdx"), vdx_update[diff_idx.numel():]]) # On ajoute la nouvelle node index
vdx[diff_idx] = vdx_update[:diff_idx.numel()] #On change les diff index

In [229]:
from torch_geometric.nn.pool.pool import pool_edge


In [ ]:
edges_coarse = torch.empty((2, 0), dtype=torch.long, device=x_f.device)

Il n'y a pas d'edge_index dans event new.

On recréé donc un coarse graph et on update le graphe avant coarsening.

In [234]:
graph_out = Data(x=x_max, pos=pos_mean, edge_index=edges_coarse)
async_mp_layer.asy_graph = Data(x=x_f, pos=pos, edge_index=edge_index, vdx=vdx).clone()

In [235]:
async_mp_layer.asy_node_max_index = torch.flatten(node_index_scatter[argmax]).long()
async_mp_layer.asy_voxel_pos_sum = voxel_pos_sum
async_mp_layer.asy_voxel_node_count = voxel_node_count
async_mp_layer.asy_graph_coarse = Data(x=graph_out.x, pos=graph_out.pos, edge_index=edges_coarse, vdx=clusters_unique)


In [236]:
#On considère dans l'asyncronous processing que les event sont dans le même batch 
graph_out.batch = torch.zeros(graph_out.num_nodes, dtype=torch.long, device=graph_out.x.device)

In [ ]:
#On assigne les positions du coarse graph aux autres 
async_mp_layer.asy_pass_attribute('asy_pos', graph_out.pos)

In [242]:
graph_out.pos.shape

torch.Size([29, 2])